# Still need to do:
1. label
2. Mask strategy to be confirmed

# Consider:
0. Load data
1. Sample 1/5 data, extract subsequences of fixed length
2. Sample to sub sequences
3. View Invariant + Normalize 
(4. Impute? Not yet. Apply behaveMAE on raw dataset )
Shape: 
```
{
    "CP1A":{
        "data":(16 * 48, 300, 10, 3)
        "drug":[CP, V, ...] * 16,
        "concentration":[0.01, 0] * 16,
     } 
     "CP1B": {
         "data":(24 * 48, 300, 10, 3)
     }
     "INH1": {
         "data":(28 * 26, 300, 10, 3)
     }
     "INH2": {
         "data":(72 * 26, 300, 10, 3)
         "drug":[AM, V, PF, MJ, AMPF, AMMJ...]
      }
      "MOS1aD": {
         "data":(24 * 26, 300, 10, 3)
         "drug":[...]
      }
}
```
5144 sub sequences samples with length 300

In [5]:
import os
import pickle
import numpy as np
from pathlib import Path
from utils.utils import PreprocessPipeline

# CLB

## Data

In [6]:
path_to_data_dir = "/home/rguo_hpc/myfolder/code/mocap/data/"
datasets = ["CP1A", 
            #"CP1B", "INH1", "INH2", "MOS1aD"
            ]
for dataset in datasets: 
    print(dataset)
    filepath = Path(path_to_data_dir + dataset + "_CLB.pkl")
    pipeline = PreprocessPipeline(
        path_to_data =  filepath,
        left_idx      = 3,
        right_idx     = 8,
        sampling_rate   = 5,
        num_frames      = 300,
        sliding_window  = 150,
        index_frame     = 149,      # int(length_input_seq / 2)
        normalizer      = 'normal', # 'cube' or 'normal'
    )
    pipeline.run()

CP1A


In [7]:
pipeline.keypoints_ids[::2]

[(0, np.int64(0)),
 (0, np.int64(300)),
 (0, np.int64(600)),
 (0, np.int64(900)),
 (0, np.int64(1200)),
 (0, np.int64(1500)),
 (0, np.int64(1800)),
 (0, np.int64(2100)),
 (0, np.int64(2400)),
 (0, np.int64(2700)),
 (0, np.int64(3000)),
 (0, np.int64(3300)),
 (0, np.int64(3600)),
 (0, np.int64(3900)),
 (0, np.int64(4200)),
 (0, np.int64(4500)),
 (0, np.int64(4800)),
 (0, np.int64(5100)),
 (0, np.int64(5400)),
 (0, np.int64(5700)),
 (0, np.int64(6000)),
 (0, np.int64(6300)),
 (0, np.int64(6600)),
 (0, np.int64(6900)),
 (1, np.int64(150)),
 (1, np.int64(450)),
 (1, np.int64(750)),
 (1, np.int64(1050)),
 (1, np.int64(1350)),
 (1, np.int64(1650)),
 (1, np.int64(1950)),
 (1, np.int64(2250)),
 (1, np.int64(2550)),
 (1, np.int64(2850)),
 (1, np.int64(3150)),
 (1, np.int64(3450)),
 (1, np.int64(3750)),
 (1, np.int64(4050)),
 (1, np.int64(4350)),
 (1, np.int64(4650)),
 (1, np.int64(4950)),
 (1, np.int64(5250)),
 (1, np.int64(5550)),
 (1, np.int64(5850)),
 (1, np.int64(6150)),
 (1, np.int64(6450)

In [ ]:
# Load each of processed one
path_to_data_dir = "/home/rguo_hpc/myfolder/code/mocap/data/"
datasets = ["CP1A", "CP1B", "INH1", "INH2", "MOS1aD"]
raw_data = []
drug_label = []
concentration_label = []

for dataset in datasets: 
    filepath = Path(path_to_data_dir + dataset + "_CLB_processed.pkl")
    with open(filepath, 'rb') as file:
        data = pickle.load(file)
    print(data["data"].shape)
    raw_data.append(data["data"])
    drug_label = drug_label + data["drug"]
    concentration_label = concentration_label + data["concentration"]

(752, 300, 10, 3)
(1120, 300, 10, 3)
(700, 300, 10, 3)
(1800, 300, 10, 3)
(600, 300, 10, 3)


In [40]:
raw_data = np.concatenate(raw_data)
print(raw_data.shape)

processed_data = {
    "data":raw_data,
    "drug": drug_label,
    "concentration": concentration_label
}


with open("/home/rguo_hpc/myfolder/code/mocap/data/data_clb.pkl", 'wb') as file:
    pickle.dump(processed_data, file)

(4972, 300, 10, 3)


In [10]:
len(data["data"])

4972

## Label

In [7]:
import pickle
with open("/home/rguo_hpc/myfolder/code/mocap/data/data_clb.pkl", 'rb') as file:
    data = pickle.load(file)

In [ ]:
import numpy as np
data = np.load("/home/rguo_hpc/myfolder/code/mocap/outputs/mocap/test_submission_1.npy", allow_pickle=True)

In [1]:
import pickle
with open("/home/rguo_hpc/myfolder/code/mocap/data/data_clb.pkl", 'rb') as file:
    y = pickle.load(file)["drug"]

# FL2